# Applying Models for Code Generation (OpenAI Codex, GitHub Copilot)

## 📚 Learning Objectives

By completing this notebook, you will:
- Use OpenAI Codex for code generation
- Use GitHub Copilot
- Generate code from descriptions
- Automate coding tasks
- Evaluate generated code

## 🔗 Prerequisites

- ✅ Understanding of code generation
- ✅ Programming knowledge
- ✅ API knowledge

---

This notebook covers practical activities from **Course 10, Unit 4**:
- Applying models like OpenAI Codex or GitHub Copilot for code generation

---

## Introduction

**Code generation models** like Codex and Copilot assist developers by generating code from natural language descriptions or context.

## 📥 Inputs & 📤 Outputs

**Inputs:** What we use in this notebook

- Libraries and concepts as introduced in this notebook; see prerequisites and code comments.

**Outputs:** What you'll see when you run the cells

- Printed results, figures, and summaries as shown when you run the cells.

---

In [1]:
print("✅ Libraries imported!")
print("\nCode Generation Models")
print("=" * 60)

print("\nOpenAI Codex:")
print("  - Code generation")
print("  - Multiple languages")
print("  - Natural language input")
print("  - API access")

print("\nGitHub Copilot:")
print("  - IDE integration")
print("  - Context-aware")
print("  - Real-time suggestions")
print("  - Code completion")

print("\nApplications:")
print("  - Code completion")
print("  - Function generation")
print("  - Bug fixing")
print("  - Documentation")

print("\n✅ Code generation concepts understood!")

✅ Libraries imported!

Code Generation Models

OpenAI Codex:
  - Code generation
  - Multiple languages
  - Natural language input
  - API access

GitHub Copilot:
  - IDE integration
  - Context-aware
  - Real-time suggestions
  - Code completion

Applications:
  - Code completion
  - Function generation
  - Bug fixing
  - Documentation

✅ Code generation concepts understood!


## 🌍 Real-World Worked Example — Character-Level Text Generator

**Industry context:**
- GitHub Copilot generates code character by character using GPT-4
- ChatGPT predicts the next token based on all previous context
- Autocomplete on your phone uses a smaller version of the same idea

We build a **character-level language model** that learns to generate text token by token — the exact mechanism behind all LLMs.

In [2]:
import torch, torch.nn as nn, torch.optim as optim
import numpy as np

torch.manual_seed(42)
# ── Training text ────────────────────────────────────────────────────────────
text = (
    "to be or not to be that is the question whether tis nobler in the mind "
    "to suffer the slings and arrows of outrageous fortune or to take arms against "
    "a sea of troubles and by opposing end them to die to sleep no more and by "
    "a sleep to say we end the heartache and the thousand natural shocks that "
    "flesh is heir to tis a consummation devoutly to be wished to die to sleep"
)

chars  = sorted(set(text))
c2i    = {c:i for i,c in enumerate(chars)}
i2c    = {i:c for c,i in c2i.items()}
VOCAB  = len(chars)
enc    = [c2i[c] for c in text]

SEQ_LEN = 20
X_list, y_list = [], []
for i in range(len(enc)-SEQ_LEN-1):
    X_list.append(enc[i:i+SEQ_LEN])
    y_list.append(enc[i+SEQ_LEN])
X_t = torch.tensor(X_list, dtype=torch.long)
y_t = torch.tensor(y_list, dtype=torch.long)

# ── LSTM Language Model ───────────────────────────────────────────────────
class CharLM(nn.Module):
    def __init__(self):
        super().__init__()
        self.embed = nn.Embedding(VOCAB, 32)
        self.lstm  = nn.LSTM(32, 128, batch_first=True, num_layers=2)
        self.fc    = nn.Linear(128, VOCAB)
    def forward(self, x):
        out,_ = self.lstm(self.embed(x))
        return self.fc(out[:,-1,:])

model   = CharLM()
opt     = optim.Adam(model.parameters(), lr=3e-3)
loss_fn = nn.CrossEntropyLoss()

for epoch in range(200):
    model.train()
    perm = torch.randperm(len(X_t))[:256]  # mini-batch
    loss = loss_fn(model(X_t[perm]), y_t[perm])
    opt.zero_grad(); loss.backward(); opt.step()
    if epoch % 50 == 0:
        print(f"Epoch {epoch} — loss: {loss.item():.3f}")

# ── Text Generation (Greedy / Temperature Sampling) ──────────────────────
def generate(seed_str, steps=80, temperature=0.8):
    model.eval()
    chars_out = list(seed_str)
    ctx = [c2i.get(c, 0) for c in seed_str[-SEQ_LEN:]]
    for _ in range(steps):
        inp = torch.tensor([ctx[-SEQ_LEN:]]).long()
        with torch.no_grad():
            logits = model(inp)[0] / temperature
        probs = torch.softmax(logits, 0).numpy()
        next_c = np.random.choice(len(probs), p=probs)
        chars_out.append(i2c[next_c])
        ctx.append(next_c)
    return ''.join(chars_out)

print("\n── Generated Text ──────────────────────────────────────────────")
print(generate("to be or not", steps=100))
print("\nThis is exactly how ChatGPT generates text — one token at a time.")

Epoch 0 — loss: 3.169


Epoch 50 — loss: 1.298


Epoch 100 — loss: 0.053


Epoch 150 — loss: 0.011



── Generated Text ──────────────────────────────────────────────
to be or notem a slea hoer sa a sin to slee and to be sopes and ir to slee and by oppos ininnuturya no wis is in

This is exactly how ChatGPT generates text — one token at a time.


## 📚 References & Further Reading

**Papers:**
- Radford et al. (2019) — [GPT-2: Language Models are Unsupervised Multitask Learners](https://d4mucfpksywv.cloudfront.net/better-language-models/language_models_are_unsupervised_multitask_learners.pdf)
- Brown et al. (2020) — [GPT-3](https://arxiv.org/abs/2005.14165)

**Interactive:**
- [Karpathy's nanoGPT](https://github.com/karpathy/nanoGPT) — build GPT in 300 lines
- [The Unreasonable Effectiveness of RNNs](http://karpathy.github.io/2015/05/21/rnn-effectiveness/)

**State-of-the-Art:** GPT-4, Claude 3.5, Gemini 1.5 — all trained on trillions of tokens with transformer decoders.

## 📝 Summary

You learned about **AI code generation** — from Codex to GitHub Copilot to modern models like DeepSeek-Coder. These models are trained on code+text corpora and fine-tuned with RLHF for helpfulness. Retrieval-Augmented Generation (RAG) over codebases further improves relevance.